# HG4052 · Week 4 Practical
## Thirteen honest numbers

**No installs: librosa, scipy and every audio tool used here is preinstalled on Colab. Headphones on again: two of this week's lessons arrive by ear.**

By the end you will have:
- ✅ one glottal period measured by hand, and 1/T checked against Praat's pitch track
- ✅ the click of a hard frame cut heard, and the Hamming taper that removes it
- ✅ the 26-triangle mel filterbank drawn, and applied as one matrix multiply
- ✅ 13 MFCCs computed by your own DCT line, matching `librosa.feature.mfcc` to the last decimal
- ✅ 60 vowel tokens sorting themselves into three islands in MFCC space, ranked by Week 1's cosine
- ✅ (take-home) an F0 shoot-out and a shrinking-bands resynthesis experiment

**How this notebook works.** Same as every week: click a cell, press **Shift + Enter**, and read (and hear) the output underneath. Cells marked **✏️ TODO** have one small blank to fill (always one line or less). Every function is provided for you to read and run, never to write.

**Short on time?** Prioritise **Setup → Part 3 → Part 4 → Part 5**: the filterbank, the DCT and the cluster plot are the week. Part 1 is two cells on the way in, Part 2 is run-and-listen, and Parts 6 and 7 are take-home by design.

---
### Before this notebook: the Praat block

This notebook is the Colab half of the practical. The Praat half comes first, cheat sheet on the slides:

- Record yourself holding /i/, /a/ and /u/ steady for one to two seconds each (New → Record mono Sound, 44100 Hz), one recording per vowel. No mic, or opting out? The fallback vowels downloaded below stand in wherever a recording of you is used.
- Pitch-track one vowel: open it (View & Edit), tick Pitch → Show pitch, click mid-vowel and read the value (Pitch → Get pitch). Write it down; Part 1 checks your hand measurement against it.
- Save each vowel as a WAV file (Objects window: Save → Save as WAV file) for the upload below.

---
## 0 · Setup

Week 1's commands first: where are you standing, and what is here?

In [ ]:
!pwd
!ls

**Imports.** One genuinely new tool this week: `dct` from `scipy.fft`, the discrete cosine transform, the lecture's probe trick as a one-line function call (`idct` runs it backwards). Everything else you have met: `librosa`, `numpy`, `matplotlib`, the `Audio` play button, and Python's own `csv` reader for the token table in Part 5.

In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt
import librosa
from scipy.fft import dct, idct
from IPython.display import Audio, display

print("librosa", librosa.__version__, "loaded: the toolbox is open")

**Get today's data.** Five files from the course repo: three fallback vowels (synthetic voices, made for this course, and perfectly steady on purpose), plus two tables computed from synthetic tokens: `vowels_mfcc.csv` for Part 5 and `fallback_logmel.csv`, Part 4's safety net. The cell checks its own work, file by file.

In [ ]:
!mkdir -p data
!wget -q -O data/fallback_i.wav https://raw.githubusercontent.com/chenchenzi/hg4052-materials/main/week04/fallback_i.wav
!wget -q -O data/fallback_a.wav https://raw.githubusercontent.com/chenchenzi/hg4052-materials/main/week04/fallback_a.wav
!wget -q -O data/fallback_u.wav https://raw.githubusercontent.com/chenchenzi/hg4052-materials/main/week04/fallback_u.wav
!wget -q -O data/vowels_mfcc.csv https://raw.githubusercontent.com/chenchenzi/hg4052-materials/main/week04/vowels_mfcc.csv
!wget -q -O data/fallback_logmel.csv https://raw.githubusercontent.com/chenchenzi/hg4052-materials/main/week04/fallback_logmel.csv

import os
expected = {
    "fallback_i.wav": 10_000, "fallback_a.wav": 10_000, "fallback_u.wav": 10_000,
    "vowels_mfcc.csv": 1_000, "fallback_logmel.csv": 150,
}
all_ok = True
for name, min_size in expected.items():
    size = os.path.getsize(f"data/{name}") if os.path.exists(f"data/{name}") else 0
    ok = size > min_size
    all_ok = all_ok and ok
    print(("✅" if ok else "❌"), f"{name:<22} {size:>8,} bytes")

if all_ok:
    print("\n✅ All five files are ready.")
else:
    print("\n❌ At least one download failed. The same files are on NTULearn in the Week 4")
    print("   folder: download them there, then drag each into data/ via the folder icon in")
    print("   Colab's left sidebar. Stuck? Ask on the NTULearn forum.")

**Your own recordings (optional, encouraged).** If you recorded /i a u/ in the Praat block, upload them here and the notebook runs on your voice; otherwise it runs on the fallbacks. Rename your files `my_i.wav`, `my_a.wav`, `my_u.wav` first, then delete the leading `#` from the commented lines and run the cell: a file dialog opens; pick all three at once. Recorded only one vowel? Upload it and repoint just the matching variable.

In [ ]:
MY_I = "data/fallback_i.wav"        # the fallbacks: steady synthetic vowels
MY_A = "data/fallback_a.wav"
MY_U = "data/fallback_u.wav"

# from google.colab import files
# import shutil
# uploaded = files.upload()
# for name in uploaded:
#     shutil.move(name, f"data/{name}")
# MY_I, MY_A, MY_U = "data/my_i.wav", "data/my_a.wav", "data/my_u.wav"

print("Working files:", MY_I, MY_A, MY_U)

---
## 1 · One glottal period, by hand

Every number this week gets squeezed out of the waveform, so start by touching the waveform. `librosa.load(..., sr=16000)` resamples everything to the course home rate on the way in (lawfully: Week 3's anti-aliasing filter runs first), so one second is always 16,000 samples and a 25 ms frame is always 400.

The plot zooms to 30 ms in the middle of the vowel: three or four laps of one glottal cycle, the wiggle the vocal folds stamp on the air.

In [ ]:
SR = 16000                                     # the course home rate: everything below runs at 16 kHz
y_a, _ = librosa.load(MY_A, sr=SR)
print(f"{len(y_a):,} samples at {SR:,} Hz = {len(y_a) / SR:.2f} s of [a]")

mid = len(y_a) // 2
zoom = y_a[mid : mid + int(0.030 * SR)]        # 30 ms from the steady middle
t_ms = np.arange(len(zoom)) / SR * 1000

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t_ms, zoom, linewidth=0.9)
ax.set_xticks(np.arange(0, 31, 2))
ax.grid(True, linewidth=0.3, alpha=0.6)
ax.set_xlabel("time (ms) within the zoom window")
ax.set_ylabel("amplitude")
ax.set_title("30 ms mid-vowel: read off the repeat, tall peak to matching tall peak")
plt.show()

Audio(y_a, rate=SR)

In [ ]:
# ✏️ TODO: measure the period. Read the time between one tall peak and the next
# matching tall peak off the plot above, in milliseconds, as finely as the grid allows.
# Shape of the answer:   T_MS = 8.3
T_MS = ...

if T_MS is ...:
    print("⬆ fill the TODO first, then re-run")
else:
    assert 2 <= T_MS <= 20, "A speaking-voice period is 2 to 20 ms (500 down to 50 Hz); re-read the plot"
    f0_hand = 1000 / T_MS
    print(f"T = {T_MS} ms, so F0 = 1000 / {T_MS} = {f0_hand:.0f} Hz")
    print("✅ Check that against the Praat pitch value from the Praat block: within a few Hz")
    print("   is a pass; a factor of two off is an octave error, and worth keeping as evidence.")

---
## 2 · Frame + window: hear the click

The pipeline starts by cutting the sound into 25 ms frames (400 samples), one every 10 ms (160 samples: the hop). But a hard cut mid-vibration leaves a cliff at each edge, and a cliff is a click. Below, the same frame twice: cut hard, and tapered by a **Hamming window** (a raised-cosine fade-in and fade-out). One frame lasts 25 ms, too short to judge by ear, so each version is looped 20 times: any seam now repeats 40 times a second and becomes a buzz you cannot miss.

**Predict before running:** which loop will sound like the vowel, and which like the vowel plus a lawnmower?

In [ ]:
FRAME = 400                                     # 25 ms at 16 kHz
HOP = 160                                       # 10 ms at 16 kHz
frame = y_a[mid : mid + FRAME]                  # one frame from the steady middle
window = np.hamming(FRAME)

print("the hard cut, looped 20 times (the seam repeats 40 times a second):")
display(Audio(np.tile(frame, 20), rate=SR, normalize=False))
print("the same frame through the Hamming taper, looped 20 times:")
display(Audio(np.tile(frame * window, 20), rate=SR, normalize=False))

fig, ax = plt.subplots(figsize=(10, 2.8))
ax.plot(np.tile(frame, 3), linewidth=0.7, label="hard cut: a cliff at every seam")
ax.plot(np.tile(frame * window, 3), linewidth=0.7, label="Hamming: faded to zero at every seam")
ax.set_xlabel("sample")
ax.set_ylabel("amplitude")
ax.set_title("three loops of each: find the cliffs")
ax.legend(loc="upper right", fontsize=8)
plt.show()

In [ ]:
# What the cliff does to the spectrum: the same frame's spectrum both ways.
spec_hard = 20 * np.log10(np.abs(np.fft.rfft(frame, n=4096)) + 1e-9)
spec_hamm = 20 * np.log10(np.abs(np.fft.rfft(frame * window, n=4096)) + 1e-9)
freqs = np.linspace(0, SR / 2, len(spec_hard))

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(freqs, spec_hard, linewidth=0.8, label="hard cut (a rectangular window)")
ax.plot(freqs, spec_hamm, linewidth=0.8, label="Hamming taper")
ax.set_xlim(0, 4000)
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("level (dB)")
ax.set_title("the cliff smears energy between the harmonics; the taper keeps them clean")
ax.legend(loc="upper right", fontsize=8)
plt.show()

**✏️ TODO (in words, one sentence).** Between the two loops you heard and the two spectra you plotted: what did the Hamming taper remove, and where had that energy been hiding in the spectrum? (Double-click to edit.)

> **...**

From here on, every frame in this notebook (and inside librosa) is tapered before its spectrum is taken. (librosa's default taper is the Hann window, the Hamming's near-identical cousin.)

---
## 3 · The mel filterbank: 26 triangles, one matrix

The ear does not measure frequency with a linear ruler; the mel scale is the ear's ruler (m = 1127 · ln(1 + f/700)). `librosa.filters.mel` builds 26 triangular band-collectors, equally spaced in mels: crowded and narrow below 1 kHz, where formants live, wide and sparse up top. Stacked as rows they are a **matrix**: 26 rows (one per triangle) by 257 columns (one per STFT frequency bin at n_fft 512).

(Notice the triangles shrink as they widen: librosa scales each one by its bandwidth, the "slaney" normalisation, so the wide top triangles do not out-shout the narrow bottom ones.)

In [ ]:
N_FFT = 512
N_MELS = 26
melfb = librosa.filters.mel(sr=SR, n_fft=N_FFT, n_mels=N_MELS, fmax=8000)
print("the filterbank is a matrix of shape", melfb.shape, "(26 triangles x 257 frequency bins)")

fft_freqs = np.linspace(0, SR / 2, melfb.shape[1])
fig, ax = plt.subplots(figsize=(10, 3.2))
for row in melfb:
    ax.plot(fft_freqs, row, linewidth=0.8)
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("filter weight")
ax.set_title("26 mel triangles: dense where the ear is precise, sparse where it is not")
plt.show()

In [ ]:
# The matrix moment. One frame's power spectrum (257 numbers) goes in;
# 26 numbers come out, one per triangle: how much energy fell into each band.
power_spec = np.abs(np.fft.rfft(frame * window, n=N_FFT)) ** 2
print("power spectrum of one Hamming-tapered frame:", power_spec.shape)

# ✏️ TODO: apply the filterbank. One matrix multiply: the @ operator.
# Shape of the answer:   mel_frame = melfb @ power_spec
mel_frame = ...

if mel_frame is ...:
    print("⬆ fill the TODO first, then re-run")
else:
    assert mel_frame.shape == (26,), "Expected 26 numbers, one per triangle: matrix (26, 257) @ vector (257,)"
    print("✅ (26, 257) @ (257,) = (26,): the whole mel front end is one matrix multiply.")
    print("   Loudest band:", int(np.argmax(mel_frame)) + 1, "of 26 (bands count from 1 here)")
    fig, ax = plt.subplots(figsize=(8, 2.8))
    ax.bar(np.arange(1, 27), mel_frame)
    ax.set_xlabel("mel band")
    ax.set_ylabel("band energy")
    ax.set_title("one frame, heard through 26 triangles")
    plt.show()

In [ ]:
# The same multiply, once per frame, is the whole mel spectrogram. Below: the vowel
# through a linear ruler (what Praat drew last week) and through the mel ruler
# (what the recognizer reads), with the P&B [a] formant means marked on both.
stft_power = np.abs(librosa.stft(y_a, n_fft=N_FFT, hop_length=HOP)) ** 2
mel_spec = melfb @ stft_power                   # the matrix, applied to every frame at once

mel_lib = librosa.feature.melspectrogram(y=y_a, sr=SR, n_fft=N_FFT, hop_length=HOP,
                                         n_mels=N_MELS, fmax=8000)
print("identical to librosa.feature.melspectrogram?", np.allclose(mel_spec, mel_lib),
      f"(largest gap {np.abs(mel_spec - mel_lib).max():.2e})")

times = np.arange(stft_power.shape[1]) * HOP / SR
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))

db_lin = librosa.power_to_db(stft_power)
ax1.pcolormesh(times, np.linspace(0, SR / 2, stft_power.shape[0]), db_lin,
               cmap="magma", vmin=db_lin.max() - 60, vmax=db_lin.max(), shading="auto")
for hz in (730, 1090):
    ax1.axhline(hz, color="#5ED0C0", linestyle="--", linewidth=1)
ax1.set_xlabel("time (s)")
ax1.set_ylabel("frequency (Hz)")
ax1.set_title("linear ruler: 257 even bins; the formants crowd the bottom", fontsize=10)

db_mel = librosa.power_to_db(mel_spec)
ax2.pcolormesh(times, np.arange(1, N_MELS + 1), db_mel,
               cmap="magma", vmin=db_mel.max() - 60, vmax=db_mel.max(), shading="auto")
for band in (7, 10):                            # where 730 and 1090 Hz land on the mel ruler
    ax2.axhline(band, color="#5ED0C0", linestyle="--", linewidth=1)
ax2.set_xlabel("time (s)")
ax2.set_ylabel("mel band")
ax2.set_title("mel ruler: 26 bands, most of them spent below 2 kHz", fontsize=10)

plt.tight_layout()
plt.show()

---
## 4 · Log + DCT: the thirteen numbers

Two steps left on the roadmap. **Log**, because loudness is log (Week 3's dB): `librosa.power_to_db` turns band energies into decibels. Then the **DCT**, the lecture's probe trick run on the 26 log-mel values: each cosine probe scores the frame with one dot product; slow probes trace the formant hills, fast probes chase the harmonic pickets. Keep the first 13 scores and you are holding the frame's MFCCs: the thirteen honest numbers.

In [ ]:
log_mel = librosa.power_to_db(mel_spec)         # the log step, done
# 1.6 s of audio is 160 hops of 10 ms; librosa centres its frames, adding one more
print("log-mel spectrogram:", log_mel.shape, f"= 26 bands x {log_mel.shape[1]} frames")

In [ ]:
# ✏️ TODO: the DCT step, on every frame at once. scipy's dct runs down the 26 bands
# when told axis=0; type=2 and norm="ortho" match librosa's convention. Keep the
# first 13 rows with [:13].
# Building blocks, in order:   dct(   log_mel, axis=0, type=2, norm="ortho"   )[:13]
mfcc_mine = ...

if mfcc_mine is ...:
    print("⬆ fill the TODO first, then re-run")
else:
    assert mfcc_mine.shape == (13, log_mel.shape[1]), "Expected 13 coefficients per frame"
    mfcc_lib = librosa.feature.mfcc(y=y_a, sr=SR, n_mfcc=13, n_fft=N_FFT,
                                    hop_length=HOP, n_mels=N_MELS, fmax=8000)
    gap = np.abs(mfcc_mine - mfcc_lib).max()
    print(f"largest disagreement with librosa.feature.mfcc, any coefficient, any frame: {gap}")
    assert gap < 1e-3, "Expected a near-exact match: same mel matrix, same log, same DCT"
    print("✅ Your one line of scipy IS librosa.feature.mfcc. No magic left in the box.")

In [ ]:
# The payoff plot: one frame's 26 log-mel values, against what the 13 kept
# coefficients can rebuild (the DCT run backwards with the other 13 zeroed).
# Unfinished TODO above? This cell still works: it falls back to a precomputed
# frame of the fallback [a] from data/fallback_logmel.csv.

if mfcc_mine is ...:
    with open("data/fallback_logmel.csv") as f:
        reader = csv.reader(f)
        next(reader)                            # skip the m1..m26 header
        frame26 = np.array([float(v) for v in next(reader)])
    source = "fallback frame (data/fallback_logmel.csv, natural-log units)"
else:
    frame26 = log_mel[:, log_mel.shape[1] // 2]
    source = "your middle frame (dB units)"

coeffs = dct(frame26, type=2, norm="ortho")     # all 26 probe scores
kept = coeffs.copy()
kept[13:] = 0                                   # drop the fast probes
smooth = idct(kept, type=2, norm="ortho")       # rebuild from 13 numbers

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(np.arange(1, 27), frame26, "o-", linewidth=1, markersize=4,
        label="all 26 log-mel values (hills + pickets)")
ax.plot(np.arange(1, 27), smooth, linewidth=2,
        label="rebuilt from the 13 kept coefficients")
ax.set_xlabel("mel band")
ax.set_ylabel("log energy")
ax.set_title(f"keep the hills, drop the pickets: {source}", fontsize=10)
ax.legend(fontsize=8)
plt.show()

**What you are looking at.** The dotted line wiggles band to band where harmonics poke through the narrow low triangles: the pickets. The smooth line is everything 13 numbers remember: the formant hills survive, the picket ripple is gone. That is why MFCCs barely move when the same vowel is spoken at a new pitch, which is exactly what a phoneme classifier wants (and exactly the bug: a tone language keeps its meaning in the part just thrown away; Week 9 returns to this).

---
## 5 · Vowels in feature space

The class dataset: 60 synthetic vowel tokens (20 each of /i a u/, pitch and formants jittered token to token, built with the same synthesizer as the fallbacks), each already reduced to its mean 13-number MFCC vector. That is `vowels_mfcc.csv`: one row per token, columns `c1` to `c13` (`c1` is the first coefficient). If 13 numbers really do hold the vowel, 60 dots should sort themselves into three islands with no labels and no help.

In [ ]:
with open("data/vowels_mfcc.csv") as f:
    rows = list(csv.DictReader(f))

tokens = [r["token"] for r in rows]
vowels = [r["vowel"] for r in rows]
X = np.array([[float(r[f"c{k}"]) for k in range(1, 14)] for r in rows])
print("feature matrix:", X.shape, "(60 tokens x 13 MFCCs);",
      " ".join(f"[{v}] x {vowels.count(v)}" for v in "iau"))

colors = {"i": "#3E8E8E", "a": "#C44E52", "u": "#4C72B0"}
fig, ax = plt.subplots(figsize=(7, 5.2))
for v in "iau":
    idx = [k for k, vv in enumerate(vowels) if vv == v]
    ax.scatter(X[idx, 1], X[idx, 2], color=colors[v], s=45, alpha=0.8, label=f"[{v}] tokens")
ax.set_xlabel("c2 (the 2nd MFCC)")
ax.set_ylabel("c3 (the 3rd MFCC)")
ax.set_title("60 tokens, two coefficients each: the vowels sort themselves")
ax.legend()
plt.show()

In [ ]:
def cosine(a, b):                               # pasted unchanged from Week 1
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

ref = X[tokens.index("i_01")]                   # one [i] token, all 13 numbers
sims = np.array([cosine(ref, x) for x in X])

order = np.argsort(sims)[::-1]
print("most similar to i_01 .................... least similar")
print("top 5:   ", "  ".join(f"{tokens[k]} {sims[k]:.4f}" for k in order[:5]))
print("bottom 5:", "  ".join(f"{tokens[k]} {sims[k]:.4f}" for k in order[-5:]))
for v in "iau":
    idx = [k for k, vv in enumerate(vowels) if vv == v]
    print(f"mean cosine from i_01 to the [{v}] tokens: {sims[idx].mean():.4f}")

**✏️ TODO (in words, double-click).** Two observations to write down:

> The c2 x c3 scatter next to Week 1's vowel chart: what carried over, and what got bent? **...**
>
> The cosine ranking: which vowel fills the top of the list, which fills the bottom, and does the [u]-versus-[a] order match their distances on the F1 x F2 chart? **...**

The similarity machine has not changed since Week 1: same cosine, one dot product. Only the numbers being compared have changed: two formants then, thirteen MFCCs now. Next week these vectors line up in time, and the machine starts recognising words.

---
## 6 · Take-home: the F0 shoot-out

**This part and Part 7 are take-home by design: nothing in class depends on them.**

The thirteen numbers deliberately threw pitch away, so anyone who needs F0 (a tone language, an intonation study, Week 10's synthesizers) must measure it separately. Two contestants, same vowel.

**Contestant 1: autocorrelation by hand.** Slide a 40 ms snippet against itself; the first strong self-match past lag zero is one period. The dot product does the sliding: Week 3's probe trick, aimed at time instead of frequency.

In [ ]:
snippet = y_a[mid : mid + 640]                  # 40 ms, mid-vowel
ac = np.correlate(snippet, snippet, mode="full")[len(snippet) - 1 :]   # lag 0 upward

lo, hi = SR // 400, SR // 50                    # consider only 50 to 400 Hz voices
lag_best = lo + int(np.argmax(ac[lo:hi]))       # the winning lag, in samples
print(f"best self-match at lag {lag_best} samples = {lag_best / SR * 1000:.2f} ms")

fig, ax = plt.subplots(figsize=(9, 2.8))
lags_ms = np.arange(hi + 40) / SR * 1000
ax.plot(lags_ms, ac[: hi + 40], linewidth=0.9)
ax.axvline(lag_best / SR * 1000, color="#C44E52", linestyle="--", linewidth=1)
ax.set_xlabel("lag (ms)")
ax.set_ylabel("self-similarity")
ax.set_title("the snippet against itself: the dashed lag is one period")
plt.show()

In [ ]:
# ✏️ TODO: turn the winning lag into a frequency. lag_best samples last
# lag_best / SR seconds, and F0 is one over that.
# Shape of the answer:   f0_ac = SR / lag_best
f0_ac = ...

if f0_ac is ...:
    print("⬆ fill the TODO first, then re-run")
else:
    assert abs(f0_ac - SR / lag_best) < 1e-9, "Expected the sampling rate divided by the winning lag"
    print(f"✅ autocorrelation says F0 = {f0_ac:.1f} Hz")
    if T_MS is not ...:
        print(f"   your Part 1 hand measurement said {1000 / T_MS:.0f} Hz: two routes, one period")

**Contestant 2: `librosa.pyin`.** YIN is autocorrelation with better manners (a normalised difference function instead of a raw peak-hunt), and pYIN wraps it in probabilities and tracks the best pitch path through time. It returns three arrays: an F0 per frame, a voiced-or-not verdict per frame, and the verdict's probability.

In [ ]:
f0_track, voiced, voiced_prob = librosa.pyin(
    y_a, fmin=50, fmax=400, sr=SR, frame_length=1024, hop_length=HOP,
)
median_f0 = np.nanmedian(f0_track[voiced])
print(f"pyin: {int(voiced.sum())} of {len(voiced)} frames voiced, median F0 = {median_f0:.1f} Hz")

fig, ax = plt.subplots(figsize=(9, 2.8))
track_times = np.arange(len(f0_track)) * HOP / SR
ax.plot(track_times, f0_track, ".", markersize=3)
if f0_ac is not ...:
    ax.axhline(f0_ac, color="#C44E52", linestyle="--", linewidth=1,
               label=f"your autocorrelation: {f0_ac:.1f} Hz")
    ax.legend(fontsize=8)
ax.set_ylim(50, 400)
ax.set_xlabel("time (s)")
ax.set_ylabel("F0 (Hz)")
ax.set_title("the pyin track: flat is a pass for a held vowel")
plt.show()

**✏️ Take-home question (double-click).** On the fallback the two contestants should agree to within a hair: it is a synthetic vowel with metronome pitch. So run Parts 1 and 6 on your own recording (point `MY_A` at it in Setup) and hunt for a disagreement: a halved or doubled value (octave error), a creaky stretch pyin refuses to call voiced, a wobble at vowel onset. Diagnose one disagreement phonetically:

> Where the trackers disagreed, and my diagnosis: **...**

---
## 7 · Take-home: how many bands does a vowel need?

The pipeline runs backwards too. `librosa.feature.inverse.mel_to_audio` takes a mel spectrogram and conjures a waveform that could have produced it (Griffin-Lim: educated guessing for the phases the pipeline threw away). That makes the band count an audible dial: rebuild the vowel from 80 bands, then 20, then 8, and hear what a coarser ear loses. (This notebook's own 26 sits between the first two contestants.)

**Predict before running:** what dies first as the bands get scarcer: the vowel's identity, or the voice's pitch?

In [ ]:
print("the original, for reference:")
display(Audio(y_a, rate=SR))

for n_bands in [80, 20, 8]:
    mel_v = librosa.feature.melspectrogram(y=y_a, sr=SR, n_fft=N_FFT, hop_length=HOP,
                                           n_mels=n_bands, fmax=8000)
    log_mel_v = librosa.power_to_db(mel_v)      # what a recognizer would store
    rebuilt = librosa.feature.inverse.mel_to_audio(
        librosa.db_to_power(log_mel_v), sr=SR, n_fft=N_FFT, hop_length=HOP, fmax=8000,
    )
    print(f"rebuilt from {n_bands} log-mel bands per frame:")
    display(Audio(rebuilt, rate=SR))

**✏️ Take-home question (double-click).** Work through /i a u/ (swap `y_a` for loads of `MY_I` and `MY_U`, or your own recordings) and describe, in phonetic terms, what dies first as the bands get scarcer: vowel identity, pitch, voice quality? At 8 bands, which two vowels are hardest to tell apart, and does that match where their formants sit?

> What died first, and where: **...**

Post your answer (and Part 6's diagnosis) to the take-home thread on NTULearn.

---
## ✅ Done looks like

- a period read off the waveform by hand, and 1/T within a few Hz of Praat's pitch value
- the hard cut's lawnmower buzz heard, and the Hamming taper that silenced it
- the 26-triangle matrix drawn, and one frame pushed through it with a single `@`
- your DCT line matching `librosa.feature.mfcc` to the last decimal, and the payoff plot: hills kept, pickets dropped
- 60 tokens sorting into three islands in the c2 x c3 plane, and Week 1's cosine ranking every token against `i_01`
- (take-home) the F0 shoot-out and the shrinking-bands resynthesis

**What you built.** Week 3 took a sound apart into a picture; this week you rebuilt the picture into the thirteen numbers a recognizer actually reads, and every stage turned out to be one line you now own: a matrix multiply, a log, a DCT. Next week those vectors meet time: two words of different lengths, and the alignment trick that lets one recognise the other.

**Next week:** ASR I: variability, template matching, dynamic time warping.